In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv(r'customer.csv')

In [4]:
df.head()


,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


In [5]:
df = df.iloc[:,2:]

In [10]:
df.head()

,review,education,purchased
0,Average,School,No
1,Poor,UG,No
2,Good,PG,No
3,Good,PG,No
4,Average,UG,No


In [11]:
X= df.iloc[:,0:2]
Y= df.iloc[:,-1]

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,    # 20% goes to test
    random_state=42   # for reproducibility
)

In [14]:
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder(categories=[
    ['Poor', 'Average', 'Good'],   # review column
    ['School', 'UG', 'PG']        # education column
])

X_train_enc = enc.fit_transform(X_train)
X_test_enc  = enc.transform(X_test)

In [15]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test  = le.transform(y_test)

# check mappings
print("X categories:", enc.categories_)
print("Y mapping:", le.classes_)   # index = encoded value

X categories: [array(['Poor', 'Average', 'Good'], dtype=object), array(['School', 'UG', 'PG'], dtype=object)]
Y mapping: ['No' 'Yes']


### now lets learn one hot encoding 


In [20]:
df = pd.read_csv(r'cars.csv')

In [21]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [22]:
X = df.drop('selling_price', axis=1)      # features
Y = df['selling_price']                   # target

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)

on branding i want whcih have frequency >100 to make column and retst cludter them in other categorieds

In [ ]:
brand_counts = X_train['brand'].value_counts()
frequent_brands = brand_counts[brand_counts > 100].index.tolist()

print("Frequent brands (freq > 100):")
print(brand_counts[brand_counts > 100])
print(f"\nTotal frequent brands: {len(frequent_brands)}")
print(f"Brands going into 'Other': {brand_counts[brand_counts <= 100].index.tolist()}")

Frequent brands (freq > 100):
brand
Maruti        1953
Hyundai       1127
Mahindra       635
Tata           586
Toyota         391
Honda          369
Ford           320
Chevrolet      185
Renault        183
Volkswagen     154
Name: count, dtype: int64

Total frequent brands: 10
Brands going into 'Other': ['BMW', 'Skoda', 'Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi', 'Jeep', 'Lexus', 'Mitsubishi', 'Force', 'Land', 'Kia', 'Daewoo', 'MG', 'Ambassador', 'Isuzu', 'Ashok', 'Peugeot', 'Opel']


In [25]:
X_train['brand'] = X_train['brand'].apply(
    lambda x: x if x in frequent_brands else 'Other')

X_test['brand']  = X_test['brand'].apply(
    lambda x: x if x in frequent_brands else 'Other')

print("\nBrand column after bucketing (train):")
print(X_train['brand'].value_counts())


Brand column after bucketing (train):
brand
Maruti        1953
Hyundai       1127
Mahindra       635
Other          599
Tata           586
Toyota         391
Honda          369
Ford           320
Chevrolet      185
Renault        183
Volkswagen     154
Name: count, dtype: int64


drop='first'	remove first category to avoid multicollinearity 


sparse_output=False	return normal array instead of sparse matrix


handle_unknown='ignore'	safely ignore unseen categories

In [26]:
from sklearn.preprocessing import OneHotEncoder
ohe_brand = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

brand_train = ohe_brand.fit_transform(X_train[['brand']])
brand_test  = ohe_brand.transform(X_test[['brand']])
brand_cols  = ohe_brand.get_feature_names_out(['brand'])

print(f"\nBrand columns after OHE: {len(brand_cols)}")
print("Columns:", brand_cols)


Brand columns after OHE: 10
Columns: ['brand_Ford' 'brand_Honda' 'brand_Hyundai' 'brand_Mahindra'
 'brand_Maruti' 'brand_Other' 'brand_Renault' 'brand_Tata' 'brand_Toyota'
 'brand_Volkswagen']


In [27]:
# ─────────────────────────────────────────────
# Step 6: OHE on fuel (N-1) and owner (N-1)
# ─────────────────────────────────────────────
ohe_fuel = OneHotEncoder(drop='first', sparse_output=False)
fuel_train = ohe_fuel.fit_transform(X_train[['fuel']])
fuel_test  = ohe_fuel.transform(X_test[['fuel']])

ohe_owner = OneHotEncoder(drop='first', sparse_output=False)
owner_train = ohe_owner.fit_transform(X_train[['owner']])
owner_test  = ohe_owner.transform(X_test[['owner']])

# ─────────────────────────────────────────────
# Step 7: combine everything
# ─────────────────────────────────────────────
km_train = X_train[['km_driven']].values
km_test  = X_test[['km_driven']].values

X_train_final = np.hstack([brand_train, fuel_train, owner_train, km_train])
X_test_final  = np.hstack([brand_test,  fuel_test,  owner_test,  km_test])

In [28]:
X_train_final

array([[0.00e+00, 0.00e+00, 0.00e+00, ..., 0.00e+00, 0.00e+00, 2.56e+03],
       [0.00e+00, 1.00e+00, 0.00e+00, ..., 0.00e+00, 0.00e+00, 8.00e+04],
       [0.00e+00, 0.00e+00, 1.00e+00, ..., 0.00e+00, 0.00e+00, 1.50e+05],
       ...,
       [0.00e+00, 0.00e+00, 1.00e+00, ..., 0.00e+00, 0.00e+00, 3.50e+04],
       [0.00e+00, 0.00e+00, 0.00e+00, ..., 0.00e+00, 0.00e+00, 2.70e+04],
       [0.00e+00, 0.00e+00, 0.00e+00, ..., 0.00e+00, 0.00e+00, 7.00e+04]],
      shape=(6502, 18))